In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from numpy.polynomial.polynomial import polymul
from ipywidgets import FloatSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# LP -> HP DIGITAL FILTER TRANSFORMATION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.tr-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.tr-header{
    display:flex;
    justify-content:space-between;
    align-items:center;
    background:#263238;
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
}

.tr-header-title{
    font-size:19px;
    font-weight:bold;
}

.tr-badge{
    background:white;
    color:#263238;
    border-radius:16px;
    padding:4px 13px;
    font-size:15px;
    font-weight:bold;
}

.tr-strip{
    display:grid;
    grid-template-columns:1fr 1fr 1fr;
    border:1px solid #b0bec5;
    border-top:none;
    border-radius:0 0 8px 8px;
    overflow:hidden;
    margin-bottom:7px;
}

.tr-cell{
    background:#f7f9fa;
    padding:8px 10px;
    text-align:center;
    font-size:13.5px;
    border-right:1px solid #cfd8dc;
}

.tr-cell:last-child{
    border-right:none;
}

.tr-cell-title{
    font-weight:bold;
    color:#455a64;
    margin-bottom:3px;
}

.tr-result{
    width:900px;
    box-sizing:border-box;
    background:#fffdf3;
    border:1px solid #d7c676;
    border-radius:7px;
    padding:8px 12px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.tr-result-title{
    font-size:14.5px;
    font-weight:bold;
    color:#5d4037;
    margin-bottom:5px;
}

.tr-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:5px 0;
}

.tr-summary{
    width:900px;
    box-sizing:border-box;
    background:#f4f7f8;
    border:1px solid #b0bec5;
    border-radius:7px;
    padding:8px 12px;
    margin-top:7px;
    font-size:13.5px;
    line-height:1.45;
    color:#37474f;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# STATIC HEADER
# ============================================================

display(HTML("""
<div class="tr-root">

<div class="tr-header">
    <div class="tr-header-title">Digital Filter Transformation</div>
    <div class="tr-badge">LP → HP</div>
</div>

<div class="tr-strip">

    <div class="tr-cell">
        <div class="tr-cell-title">Prototype</div>
        Low-pass digital filter
    </div>

    <div class="tr-cell">
        <div class="tr-cell-title">Transformation</div>
        180° unit-circle mapping
    </div>

    <div class="tr-cell">
        <div class="tr-cell-title">Result</div>
        High-pass with a new cutoff
    </div>

</div>

</div>
"""))

# ============================================================
# PROTOTYPE FILTER
# ============================================================

Omega_p = 0.15*np.pi

b = np.array([0.008616,0.025848,0.025848,0.008616],dtype=float)

a = np.array([1.000000,-2.064414,1.519112,-0.385767],dtype=float)

# ============================================================
# CONTROLS
# ============================================================

wp_slider = FloatSlider(value=0.65,min=0.20,max=0.85,step=0.01,description='ωp/π:',continuous_update=True,readout_format='.2f',style={'description_width':'42px'},layout=Layout(width='350px'))

probe_slider = FloatSlider(value=0.65,min=0.00,max=1.00,step=0.01,description='Probe ω/π:',continuous_update=True,readout_format='.2f',style={'description_width':'75px'},layout=Layout(width='390px'))

controls = HBox([wp_slider,probe_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b0bec5',padding='7px 12px',margin='0 0 2px 0'))

# ============================================================
# RESULT PANEL
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# SUMMARY BOX
# ============================================================

summary_html = HTML("""
<div class="tr-summary">
The upper plots compare the prototype low-pass response with the transformed high-pass response.
The lower plot shows the reversed nonlinear mapping <b>ω → Ω</b>: low transformed frequencies correspond to high prototype frequencies and vice versa.
The green point marks the required cutoff correspondence <b>ω<sub>p</sub> → Ω<sub>p</sub></b>, while the red Probe point traces any selected transformed frequency back to the corresponding prototype frequency.
</div>
""")

# ============================================================
# POLYNOMIAL UTILITIES
# ============================================================

def polynomial_power(p,n):

    result = np.array([1.0])

    for _ in range(n):

        result = polymul(result,p)

    return result

def substitute_rational_polynomial(c,P,Q,N):

    result = np.zeros(N+1)

    for k,ck in enumerate(c):

        if k > N:

            break

        term = ck*polymul(polynomial_power(P,k),polynomial_power(Q,N-k))

        result[:len(term)] += term

    return result

# ============================================================
# LP -> HP TRANSFORMATION
# ============================================================

def transform_lp_to_hp(b,a,Omega_p,omega_p):

    alpha = np.cos((Omega_p-omega_p)/2)/np.cos((Omega_p+omega_p)/2)

    P = np.array([alpha,-1.0])

    Q = np.array([1.0,-alpha])

    order = max(len(b),len(a))-1

    bt_raw = substitute_rational_polynomial(b,P,Q,order)

    at_raw = substitute_rational_polynomial(a,P,Q,order)

    bt = bt_raw[::-1]

    at = at_raw[::-1]

    bt = bt/at[0]

    at = at/at[0]

    return alpha,bt,at

# ============================================================
# FREQUENCY MAPPING
# ============================================================

def frequency_mapping(alpha,omega):

    z_tilde = np.exp(1j*omega)

    z = -(z_tilde-alpha)/(1-alpha*z_tilde)

    Omega = np.abs(np.angle(z))

    return Omega

def map_single_frequency(alpha,omega):

    z_tilde = np.exp(1j*omega)

    z = -(z_tilde-alpha)/(1-alpha*z_tilde)

    Omega = abs(np.angle(z))

    return Omega

# ============================================================
# INITIAL DATA
# ============================================================

omega_p = wp_slider.value*np.pi

alpha,bt,at = transform_lp_to_hp(b,a,Omega_p,omega_p)

omega_grid = np.linspace(0,np.pi,1400)

_,H_proto = signal.freqz(b,a,worN=omega_grid)

_,H_trans = signal.freqz(bt,at,worN=omega_grid)

Omega_map = frequency_mapping(alpha,omega_grid)

probe_omega = probe_slider.value*np.pi

probe_Omega = map_single_frequency(alpha,probe_omega)

prototype_limit = 1.08*max(np.max(np.abs(H_proto)),1.0)

# ============================================================
# SINGLE FIGURE
# ============================================================

fig = plt.figure(figsize=(9.0,8.2))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

gs = fig.add_gridspec(2,2,height_ratios=[1.0,1.15],hspace=0.52,wspace=0.28)

ax_proto = fig.add_subplot(gs[0,0])

ax_trans = fig.add_subplot(gs[0,1])

ax_mapping = fig.add_subplot(gs[1,:])

fig.subplots_adjust(left=0.08,right=0.98,top=0.965,bottom=0.12,hspace=0.52,wspace=0.28)

# ============================================================
# PROTOTYPE RESPONSE
# ============================================================

proto_line, = ax_proto.plot(omega_grid/np.pi,np.abs(H_proto),linewidth=1.5,label='Prototype LP')

proto_cutoff_line = ax_proto.axvline(Omega_p/np.pi,linestyle='--',linewidth=1.1,label=r'$\Omega_p$')

proto_probe, = ax_proto.plot([probe_Omega/np.pi],[np.interp(probe_Omega,omega_grid,np.abs(H_proto))],'o',markersize=6)

ax_proto.set_xlim(0,1)

ax_proto.set_ylim(0,prototype_limit)

ax_proto.set_title('Prototype Low-Pass Response')

ax_proto.set_xlabel(r'Prototype frequency $\Omega/\pi$')

ax_proto.set_ylabel(r'$|H(e^{j\Omega})|$')

ax_proto.grid(True,linestyle=':',alpha=0.28)

ax_proto.legend(loc='upper center',bbox_to_anchor=(0.5,-0.21),ncol=2,frameon=True)

# ============================================================
# TRANSFORMED RESPONSE
# ============================================================

trans_line, = ax_trans.plot(omega_grid/np.pi,np.abs(H_trans),linewidth=1.5,label='Transformed HP')

trans_cutoff_line = ax_trans.axvline(omega_p/np.pi,linestyle='--',linewidth=1.1,label=r'$\omega_p$')

trans_probe, = ax_trans.plot([probe_omega/np.pi],[np.interp(probe_omega,omega_grid,np.abs(H_trans))],'o',markersize=6)

ax_trans.set_xlim(0,1)

ax_trans.set_ylim(0,prototype_limit)

ax_trans.set_title('Transformed High-Pass Response')

ax_trans.set_xlabel(r'New frequency $\omega/\pi$')

ax_trans.set_ylabel(r'$|\widetilde{H}(e^{j\omega})|$')

ax_trans.grid(True,linestyle=':',alpha=0.28)

ax_trans.legend(loc='upper center',bbox_to_anchor=(0.5,-0.21),ncol=2,frameon=True)

# ============================================================
# FREQUENCY MAPPING
# ============================================================

mapping_line, = ax_mapping.plot(omega_grid/np.pi,Omega_map/np.pi,linewidth=1.6,label=r'$\Omega(\omega)$')

reverse_identity_line, = ax_mapping.plot([0,1],[1,0],'--',linewidth=1.0,label='Ideal reversal')

cutoff_marker, = ax_mapping.plot([omega_p/np.pi],[Omega_p/np.pi],'o',markersize=7,label='Cutoff mapping')

probe_marker, = ax_mapping.plot([probe_omega/np.pi],[probe_Omega/np.pi],'s',markersize=6,label='Probe')

probe_vertical = ax_mapping.axvline(probe_omega/np.pi,linestyle=':',linewidth=1.0)

probe_horizontal = ax_mapping.axhline(probe_Omega/np.pi,linestyle=':',linewidth=1.0)

ax_mapping.set_xlim(0,1)

ax_mapping.set_ylim(0,1)

ax_mapping.set_title('Frequency Mapping on the Unit Circle')

ax_mapping.set_xlabel(r'Transformed frequency $\omega/\pi$')

ax_mapping.set_ylabel(r'Prototype frequency $\Omega/\pi$')

ax_mapping.grid(True,linestyle=':',alpha=0.28)

ax_mapping.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=4,frameon=True)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    omega_p = wp_slider.value*np.pi

    probe_omega = probe_slider.value*np.pi

    alpha,bt,at = transform_lp_to_hp(b,a,Omega_p,omega_p)

    _,H_trans = signal.freqz(bt,at,worN=omega_grid)

    trans_line.set_ydata(np.abs(H_trans))

    trans_cutoff_line.set_xdata([omega_p/np.pi,omega_p/np.pi])

    Omega_map = frequency_mapping(alpha,omega_grid)

    probe_Omega = map_single_frequency(alpha,probe_omega)

    mapping_line.set_ydata(Omega_map/np.pi)

    cutoff_marker.set_data([omega_p/np.pi],[Omega_p/np.pi])

    probe_marker.set_data([probe_omega/np.pi],[probe_Omega/np.pi])

    probe_vertical.set_xdata([probe_omega/np.pi,probe_omega/np.pi])

    probe_horizontal.set_ydata([probe_Omega/np.pi,probe_Omega/np.pi])

    prototype_probe_value = np.interp(probe_Omega,omega_grid,np.abs(H_proto))

    transformed_probe_value = np.interp(probe_omega,omega_grid,np.abs(H_trans))

    proto_probe.set_data([probe_Omega/np.pi],[prototype_probe_value])

    trans_probe.set_data([probe_omega/np.pi],[transformed_probe_value])

    mapped_cutoff = map_single_frequency(alpha,omega_p)

    mapping_error = abs(mapped_cutoff-Omega_p)

    poles = np.roots(at)

    maximum_pole_radius = np.max(np.abs(poles))

    b_text = ', '.join([f'{value:.6f}' for value in bt])

    a_text = ', '.join([f'{value:.6f}' for value in at])

    result_html.value = f"""
    <div class="tr-result">

    <div class="tr-result-title">
    Current LP → HP transformation
    </div>

    Prototype cutoff:
    <b>Ω<sub>p</sub> = {Omega_p/np.pi:.3f}π</b>

    &nbsp;&nbsp;&nbsp;

    New cutoff:
    <b>ω<sub>p</sub> = {omega_p/np.pi:.3f}π</b>

    &nbsp;&nbsp;&nbsp;

    Transformation parameter:
    <b>α = {alpha:.6f}</b>

    <div class="tr-equation">
    z =
    −(z̃ − α)/(1 − αz̃)
    </div>

    Probe mapping:

    <b>
    ω = {probe_omega/np.pi:.3f}π
    →
    Ω = {probe_Omega/np.pi:.3f}π
    </b>

    <br>

    Cutoff mapping error:
    <b>{mapping_error:.3e} rad/sample</b>

    &nbsp;&nbsp;&nbsp;

    Maximum transformed pole radius:
    <b>{maximum_pole_radius:.6f}</b>

    <br><br>

    <b>Transformed numerator:</b>
    [{b_text}]

    <br>

    <b>Transformed denominator:</b>
    [{a_text}]

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# OBSERVERS
# ============================================================

wp_slider.observe(update,names='value')

probe_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(controls)

display(fig.canvas)

display(summary_html)

update()